##1. Environment parameters

In [0]:
#Naming convention: <catalog>.<schema>.<table>, adopting snake_case 

catalog = "cinedata_medallion"
land_schema_name = "landing"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

land_schema = f"{catalog}.{land_schema_name}"
bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path =  f"/Volumes/{catalog}/{land_schema_name}/inputs"

print(f"catalog: {catalog}")
print(f"land_schema: {land_schema}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

##2. Catalog, schema and volume creation

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")          
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {land_schema}")    
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")     
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")     
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")       
spark.sql(f"CREATE VOLUME IF NOT EXISTS {land_schema}.inputs")  

print("Catalog, schemas & volume done.")

##3. Landing zone validation

In [0]:
expected_files = [
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_info_TMDB_IMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "movies_reviews.csv"
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERT] Couldn't list files in '{landing_path}'. Upload the files to continue.\n{e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("[PENDING] Files not found inside landing zone:")
    for f in missing:
        print(f"  - {f}")
else:
    print("[OK] All the expected files are in landing zone:")
    for f in dbutils.fs.ls(landing_path):
        print(f"  - {f.name} ({f.size/1024:.1f} KB)")